# Lab 62: The Facial Feedback

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-62.ipynb)

**What you will do:** compare your own teeth-versus-lips funniness ratings, pool the class's data, and check whether a pretrained facial-expression classifier reads your two selfies differently.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

This Lab's smartphone extension asks you to take selfies in each pen condition. Keep them on
your own device for your own comparison; do not upload or share them with classmates or in any
pooled dataset. When you pool class data below, share only the numeric funniness ratings, never
the images.

## The experiment in the book

In Lab 62 you held a pen between your teeth (forcing a smile-like muscle contraction) while
rating cartoons for funniness, then between your lips (blocking that contraction) while rating a
different set. In the classic study (Strack, Martin, & Stepper, 1988), the teeth condition
produced funnier ratings than the lips condition. The book is careful to flag that this finding
is contested: a large, strict multi-lab replication (Wagenmakers et al., 2016) found essentially
no effect, while a later, much larger effort — the Many Smiles Collaboration (Coles et al.,
2022), almost 3,900 participants in 19 countries — found a real but small effect that depended
on exactly how the expression was produced, with the original pen-in-mouth method giving the
weakest and least consistent result of the three methods tested.

## Record your results

Enter your average funniness rating (1-10) for the cartoons you rated in each condition.

In [ ]:
results = pd.DataFrame({
    "condition": ["teeth (smile-like)", "lips (blocked)"],
    "mean_funniness": [6.4, 5.8],   # example data: replace with your own average ratings
})
display(results)
results.plot.bar(x="condition", y="mean_funniness", legend=False, color=["steelblue", "grey"], figsize=(4.5, 3))
plt.ylabel("mean funniness rating (1-10)"); plt.ylim(0, 10); plt.xticks(rotation=0); plt.show()

## Compare

The lab does not give a specific effect size in points or percent for this contested finding —
only that the original study found teeth funnier than lips, that a strict replication found
"essentially no effect," and that the largest effort found a real but small effect that
depended heavily on the exact method. So the honest comparison here is about *direction*, not
magnitude, and about how small a difference this literature actually expects.

In [ ]:
diff = results.set_index("condition").loc["teeth (smile-like)", "mean_funniness"] \
     - results.set_index("condition").loc["lips (blocked)", "mean_funniness"]
print(f"Your teeth-minus-lips difference: {diff:+.1f} points")
if diff > 0:
    print("Your data points the same direction as the original 1988 study. Given that the")
    print("largest replication (Coles et al., 2022) found this effect to be small and method-")
    print("dependent, do not expect a large or highly reliable gap in a one-person trial.")
elif diff == 0:
    print("No difference at all -- consistent with the strict Wagenmakers et al. (2016)")
    print("replication, which found essentially no effect for this exact pen procedure.")
else:
    print("Your data ran the opposite direction. Given how small and inconsistent this effect")
    print("has proven across thousands of participants, a reversal in one person's ratings is")
    print("not surprising and does not, on its own, contradict the wider literature.")

## The AI side

A pretrained facial-expression classifier can rate how "happy" a face looks from a still
image, independent of how the expression was produced. Upload your two selfies (teeth
condition, lips condition) from the smartphone extension and compare the model's happiness
score for each.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline

try:
    from google.colab import files
    print("Upload two selfies named teeth.jpg and lips.jpg.")
    uploaded = list(files.upload().keys())
except ImportError:
    uploaded = []  # not running in Colab

clf = pipeline("image-classification", model="trpakov/vit-face-expression")
rows = []
for fname in uploaded:
    condition = fname.rsplit(".", 1)[0].lower()
    preds = {p["label"].lower(): p["score"] for p in clf(fname)}
    rows.append({"file": fname, "condition": condition, "happy_score": round(preds.get("happy", 0.0), 2)})
ai_df = pd.DataFrame(rows)
display(ai_df)

This classifier reads only the static geometry of your face; it has no access to which
muscles you contracted on purpose or how you felt. If it rates the teeth photo as "happier,"
that shows the pen changed your visible expression, which is a different (and much less
contested) claim than the facial *feedback* hypothesis — that the expression also changed how
funny the cartoons felt to you. The two effects are related but not the same, and a small
pretrained model's percentage scores should be read as rough, illustrative signals rather than
precise measurements.

## Pool the class data

Pool the class's mean funniness ratings for each condition (anonymous IDs, not names, and no
images).

In [ ]:
import io
csv_text = """id,condition,mean_funniness
P01,teeth,6.5
P01,lips,5.9
P02,teeth,7.0
P02,lips,7.1
P03,teeth,5.8
P03,lips,5.2
"""  # example data: replace with your class CSV, e.g. pd.read_csv("facial_feedback_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
wide = df.pivot(index="id", columns="condition", values="mean_funniness")
wide["diff"] = wide["teeth"] - wide["lips"]
display(wide)
boot = [wide["diff"].sample(frac=1, replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean teeth-minus-lips difference: {wide['diff'].mean():+.2f} "
      f"(95% bootstrap interval {lo:+.2f} to {hi:+.2f}, n = {len(wide)})")
print("The largest published replication (Coles et al., 2022) found this effect to be small")
print("and dependent on method, so do not expect the class's small sample to settle the debate.")

## Questions to think about

1. Did your own teeth-minus-lips difference match the direction of the original 1988 finding? Given the replication history, how much weight should a single person's result carry either way?
2. The Many Smiles Collaboration found the pen-in-mouth method gave the weakest result of the three methods it tested. Why might physically holding a pen in your mouth interfere with the effect it is meant to produce, compared with just posing a smile on request?
3. If the AI classifier's happy score differed between your two selfies, does that tell you anything about whether the cartoons actually felt funnier, or only about how your face looked?
4. William James argued "we are sad because we cry" rather than the reverse. Can you think of a bodily action (other than smiling) that you use, deliberately or not, to try to change how you feel?

## Challenge

Try one of the two methods that performed better in the Many Smiles Collaboration: ask a friend
to either copy a photographed smiling expression, or simply pose a smile on request (rather than
holding a pen), and rate the same cartoons. Compare the size of the effect with your pen-based
result above.

In [ ]:
# Example data: replace with your own posed-smile comparison.
posed = pd.DataFrame({
    "condition": ["posed smile", "neutral face"],
    "mean_funniness": [7.2, 5.9],   # example data
})
pen_diff = 0.6      # your teeth-minus-lips difference from the record section, for comparison
posed_diff = posed.set_index("condition").loc["posed smile", "mean_funniness"] \
           - posed.set_index("condition").loc["neutral face", "mean_funniness"]
print(f"Pen-in-mouth difference:  {pen_diff:+.1f}")
print(f"Posed-smile difference:   {posed_diff:+.1f}")
print("The Many Smiles Collaboration found posed and copied smiles gave a more consistent effect")
print("than the pen-in-mouth method. The numbers above are example data -- use your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-62.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")